In [ ]:
m = pybma.load_model("BMA_model.json")
# pybma.save_model(m,"B-diff-pyBMA.json")
qn = pybma.model_to_qn(m)
p = pybma.check_stability(qn)

nameMap = pybma.model_to_variableIDdict(m)



In [ ]:
def B_cell_simulation(model, combinaison, Formulas, timing, output_dir, prefix = 'BCRABL', nameMap = None, ):
    """
    Export simulation results to CSV using combination parameters for naming.

    This function iterates through gene combinations, constructs a filename
    based on the gene, timing stage, and mutation status, and writes the
    corresponding simulation data to a CSV file.

    Args:
        combinaison: Dictionary containing gene groupings and statuses.
        timing: Dictionary mapping time indices to stage names (e.g., {0: 'CLP'}).
        output_dir: The directory where the CSV files will be saved.
        nameMap: Dictionary mapping node ID to thier name
        prefix: The leading string for the filename.
    """
    # Ensure a directory exists for the outputs
 # Initialize storage and directory
    simulations = []
    all_combinaisons = []
    genes_lists = []

    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    for primary_gene, groups in combinaison.items():
        for entry in groups:
            
            sub_genes = list(entry.keys())
            
            # 1. Generate combinations (Ensure single genes are tuples)
            if len(sub_genes) > 1:
                raw_combinations = generate_all_gene_combinations(len(sub_genes), 4)
            else:
                # Standardizes format to list of tuples: [(0,), (1,), (2,), (3,)]
                raw_combinations = [(0,), (1,), (2,), (3,)]
                
            # 2. Map genes to their timing values
            genes_list = [dict(zip(sub_genes, combo)) for combo in raw_combinations]
            
            # Track for debugging/history
            genes_lists.append(genes_list)
            all_combinaisons.append(raw_combinations)
    
            # 3. Run Simulation Loop
            for t in range(len(genes_list)):
                current_combo_dict = genes_list[t] # The dict: {'Ikzf1': 0}
                current_combo_tuple = raw_combinations[t] # The tuple: (0,)
                
                mod = copy.deepcopy(model)    
                var_lookup = {var["Name"]: var for var in mod["Model"]["Variables"]}
                var_lookup['BCR-ABL']['RangeTo'] = 2
                
                for gene_name, status in entry.items():
                    if gene_name in var_lookup:
                        target_var = var_lookup[gene_name]
                        
                        # Get timing value and handle tuple vs int
                        val = current_combo_dict[gene_name]
                        timing_index = val[0] if isinstance(val, tuple) else val
                        
                        if gene_name in ('Ikzf1', 'Pax5'):
                            node_to_update = Formulas[gene_name]['Node']
                            target_formula_list = Formulas[gene_name]['Formula']
                            if Formulas[gene_name][status] == 1:
                                var_lookup[node_to_update]['Formula'] = 'min(1,' +  target_formula_list[timing_index]+')'
                            else:
                                var_lookup[node_to_update]['Formula'] =  target_formula_list[timing_index]
                                # var_lookup[node_to_update]['RangeTo'] = Formulas[gene_name][status]
                             
                        else:
                            node_to_update = Formulas[gene_name]['Node']
                            target_formula_list = Formulas[gene_name]['Formula']
                            var_lookup[node_to_update]['Formula'] = target_formula_list[timing_index]  

                    qn2 = pybma.model_to_qn(mod)
                    sim_result = pybma.simulate(qn2, steps=30)
                    simulations.append(sim_result)
        
                # 4. Filename Generation
                name_parts = [prefix]
                for i, gene in enumerate(sub_genes):
                    # Access the tuple index safely
                    t_idx = current_combo_tuple[i]
                    time_stage = timing.get(t_idx, "Unknown")
                    mut_status = entry[gene].upper()
                    name_parts.extend([gene.upper(), time_stage, mut_status])
                
                filename = "-".join(name_parts) + ".csv"
                file_path = os.path.join(output_dir, filename)
    
                # 5. Write to CSV (Row format: [GeneID, val1, val2...])
                with open(file_path, mode='w', newline='') as f:
                    writer = csv.writer(f)
                    
                    # Apply nameMap if it exists
                    if nameMap is not None:
                        data_dict = {nameMap.get(k, k): v for k, v in sim_result.items()}
                    else:
                        data_dict = sim_result
                    
                    for key, values in data_dict.items():
                        writer.writerow(['',key] + values)

In [ ]:
combinaison = {
     'EBF1': [
          {'EBF1': 'HETERO'},
          {'EBF1': 'HETERO', 'CDKN2A': 'HOMO'},
          {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO'},
          {'EBF1': 'HETERO', 'Ikzf1': 'HOMO'}, 
          {'EBF1': 'HETERO', 'Ikzf1': 'HETERO'}, 
          {'EBF1': 'HETERO', 'Ikzf1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO'},
          {'EBF1': 'HETERO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO'},
          {'EBF1': 'HETERO', 'Ikzf1': 'HOMO', 'Pax5': 'HETERO'},
          {'EBF1': 'HETERO', 'Pax5': 'HETERO'},
          {'EBF1': 'HETERO', 'Pax5': 'HOMO', 'CDKN2A': 'HOMO'}
      ],
      'Ikzf1': [
          {'Ikzf1': 'HOMO'},
          {'Ikzf1': 'HETERO'},
          {'Ikzf1': 'HOMO', 'Pax5': 'HETERO'},
          {'Ikzf1': 'HOMO', 'CDKN2A': 'HOMO'},
          {'Ikzf1': 'HOMO', 'Pax5': 'HETERO', 'CDKN2A': 'HOMO'}
      ],
     'Pax5': [
         {'Pax5': 'HETERO'},
         {'Pax5': 'HETERO', 'CDKN2A': 'HOMO'},
         {'Pax5': 'HETERO', 'Ikzf1': 'HETERO'},
         {'Pax5': 'HETERO', 'Ikzf1': 'HETERO', 'CDKN2A': 'HOMO'}        
     ]
}                                

# Parsing logic

Formulas = { 
    'EBF1' : {'Node' : 'EBF1_KO', 'Formula' : ['var(146)+var(457)', 'var(147)+var(457)', 'var(148)+var(457)', 'var(149)+var(457)']},
    'Pax5': {'Formula' : ['var(146)+var(464)', 'var(147)+var(464)', 'var(148)+var(464)', 'var(149)+var(464)'],
                            'HOMO' : 2,'HETERO' : 1, 'Node' : 'PAX5_KO'},
    'Ikzf1' : {'Formula' : ['var(146)+var(408)', 'var(147)+var(408)', 'var(148)+var(408)', 'var(149)+var(408)'],
                            'HOMO' : 2,'HETERO' : 1, 'Node' : 'IKZF1_ko'},
    'CDKN2A' : {'Node' : 'KO_CDKN2A', 'Formula' : ['var(146)+var(425)', 'var(147)+var(425)', 'var(148)+var(425)', 'var(149)+var(425)']},
    'BCR-ABL' : {'Node' : 'KO_BCRABL', 'Formula' : ['var(146)+var(473)','var(147)+var(473)', 'var(148)+var(473)', 'var(149)+var(473)']},
    
    'JAK-STAT' : {'Node' : 'KO_JAK_STAT', 'Formula' : ['var(146)+var(392)', 'var(147)+var(392)', 'var(148)+var(392)', 'var(149)+var(392)']},
    'MAPK' : {'Node' : 'KO_MAPK', 'Formula' : ['var(146)+var(472)', 'var(147)+var(472)', 'var(148)+var(472)', 'var(149)+var(472)']},
    'PIP3' : {'Node' : 'KO_PIP3', 'Formula' : ['var(146)+var(471)', 'var(147)+var(471)', 'var(148)+var(471)', 'var(149)+var(471)']},
    'Myc': {'Node' : 'MYC_KO', 'Formula' : ['var(146)+var(471)', 'var(147)+var(415)', 'var(148)+var(415)', 'var(149)+var(415)']},
    'Bcl6': {'Node' : 'BCL6_KO', 'Formula' : ['var(146)+var(414)', 'var(147)+var(414)', 'var(148)+var(414)', 'var(149)+var(414)']}
}

Formulas_steps = { 
    'EBF1' : {'HETERO' : 'min(1,floor(((var(8)+var(59)+var(11)+var(171)+var(74))/5)+1/3)+(2*var(7)-2)-max(0,(2*var(206)-var(211)))-var(237)/2)'},
    'Pax5': {'HOMO' : '0','HETERO' : 'min(1, floor(avg(var(74),var(8)))+max(0,(2*var(11)-2))-var(275)/2 +2*var(437))'},
    'Ikzf1' : {'HOMO' : '0','HETERO' : 'min(1, 2+var(8))'},
    'CDKN2A' :{ 'HOMO' : '0'}  
}
timing = {
    0 : 'CLP',
    1 : 'PreproB',
    2 : 'ProB',
    3 : 'PreB'
}

end_state = {
    'EBF1' : ['PreproB','ProB'],
    'IKZF1' : 'ProB', 
    'Pax5' : 'PreB'
}

end_state = {
        'EBF1': ['PreproB', 'ProB'],
        'IKZF1': ['PreproB', 'ProB'],
        'Pax5': ['PreB']
    }

In [ ]:
output_dir = './our_dir_output/'
B_cell_simulation(m, combinaison, Formulas, timing, output_dir, prefix, nameMap = nameMap)